<a href="https://colab.research.google.com/github/harsha1236/code/blob/main/Fine_Tuning_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import keras
from keras.models import Sequential
from keras.layers import Dense

import warnings
warnings.filterwarnings('ignore')

In [2]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 989.4 kB/s eta 0:00:00


In [3]:
import keras_tuner
from kerastuner.tuners import RandomSearch

In [6]:
data = pd.read_csv('/content/sample_data/AQI (1).csv')
data.shape

(1093, 9)

In [7]:
data.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [8]:
x = data.iloc[:, :-1]
y = data.iloc[:, -1]

In [9]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [10]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [11]:

def build_model(hp):
    model = Sequential()

    model.add(Dense(units=8, kernel_initializer='he_uniform'))

    for i in range(hp.Int('num_layers',2, 5)):
        model.add(Dense(units=hp.Int('units_'+str(i),
                                     min_value=2,
                                     max_value=6),
                        kernel_initializer=hp.Choice('initializer', ['he_uniform', 'he_normal']),
                        activation=hp.Choice('activation', ['relu', 'leaky_relu', 'linear'])))
    model.add(Dense(1, activation='relu'))

    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])),
                  loss='mean_squared_error',
                  metrics=['accuracy'])

    return model

In [12]:
tuner = RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=2,
    overwrite=True,
    directory="my_dir",
    project_name="AQI",
)

In [13]:
tuner.search_space_summary()

Search space summary
Default search space size: 6
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 5, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 6, 'step': 1, 'sampling': 'linear'}
initializer (Choice)
{'default': 'he_uniform', 'conditions': [], 'values': ['he_uniform', 'he_normal'], 'ordered': False}
activation (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'leaky_relu', 'linear'], 'ordered': False}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 6, 'step': 1, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [14]:
tuner.search(x_train_scaled, y_train, epochs=5, validation_data=(x_test_scaled, y_test))

Trial 10 Complete [00h 00m 09s]
val_accuracy: 0.013698630034923553

Best val_accuracy So Far: 0.013698630034923553
Total elapsed time: 00h 01m 22s


In [15]:
tuner.results_summary()

Results summary
Results in my_dir/AQI
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 00 summary
Hyperparameters:
num_layers: 5
units_0: 3
initializer: he_normal
activation: leaky_relu
units_1: 6
learning_rate: 0.0001
units_2: 2
units_3: 2
units_4: 2
Score: 0.013698630034923553

Trial 01 summary
Hyperparameters:
num_layers: 5
units_0: 2
initializer: he_normal
activation: leaky_relu
units_1: 6
learning_rate: 0.0001
units_2: 2
units_3: 5
units_4: 2
Score: 0.013698630034923553

Trial 05 summary
Hyperparameters:
num_layers: 2
units_0: 5
initializer: he_uniform
activation: linear
units_1: 2
learning_rate: 0.001
units_2: 3
units_3: 6
units_4: 3
Score: 0.013698630034923553

Trial 06 summary
Hyperparameters:
num_layers: 5
units_0: 6
initializer: he_uniform
activation: relu
units_1: 3
learning_rate: 0.0001
units_2: 4
units_3: 5
units_4: 3
Score: 0.013698630034923553

Trial 09 summary
Hyperparameters:
num_layers: 5
units_0: 3
initializer: he_normal
activation: relu
